# Olist E-Commerce Dataset — Exploratory Data Analysis

This notebook explores the Brazilian E-Commerce Public Dataset by Olist.

## Objectives

- Understand the source datasets
- Identify table grain
- Identify primary and foreign keys
- Analyze data types
- Identify missing values
- Investigate relationships between tables
- Identify data quality issues
- Generate initial business insights

In [1]:
## 1. Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.__version__

'2.3.3'

In [ ]:
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs/eda")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
## 2. Load datasets

tables = {
    "customers": pd.read_csv(DATA_DIR / "olist_customers_dataset.csv"),
    "orders": pd.read_csv(DATA_DIR / "olist_orders_dataset.csv"),
    "order_items": pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv"),
    "payments": pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv"),
    "reviews": pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv"),
    "products": pd.read_csv(DATA_DIR / "olist_products_dataset.csv"),
    "sellers": pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv"),
    "geolocation": pd.read_csv(DATA_DIR / "olist_geolocation_dataset.csv"),
    "category_translation": pd.read_csv(
        DATA_DIR / "product_category_name_translation.csv"
    ),
}

tables.keys()

dict_keys(['customers', 'orders', 'order_items', 'payments', 'reviews', 'products', 'sellers', 'geolocation', 'category_translation'])

In [4]:
## 3. Dataset overview
table_summary = pd.DataFrame({
    table_name: {
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for table_name, df in tables.items()
}).T

table_summary


,rows,columns
customers,99441,5
orders,99441,8
order_items,112650,7
payments,103886,5
reviews,99224,7
products,32951,9
sellers,3095,4
geolocation,1000163,5
category_translation,71,2


In [5]:
table_summary.to_csv(OUTPUT_DIR / "table_summary.csv")


In [7]:
#inspect the schema
# for name, df in tables.items():
#     print("=" * 70)
#     print(name)
#     print("=" * 70)
#     print(df.info())

# for name, df in tables.items():
#     print(name)
#     print(df.columns.tolist())
#     print()

for name, df in tables.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(df.info())
    print("Columns:", df.columns.tolist())
    print()

customers
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
None
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

orders
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-

In [8]:
#datatypes
column_profile = []

for table_name, df in tables.items():
    for column in df.columns:
        column_profile.append({
            "table": table_name,
            "column": column,
            "dtype": str(df[column].dtype),
            "null_count": df[column].isna().sum(),
            "null_pct": df[column].isna().mean() * 100,
            "unique_count": df[column].nunique()
        })

column_profile = pd.DataFrame(column_profile)

column_profile.head()


,table,column,dtype,null_count,null_pct,unique_count
0,customers,customer_id,object,0,0.0,99441
1,customers,customer_unique_id,object,0,0.0,96096
2,customers,customer_zip_code_prefix,int64,0,0.0,14994
3,customers,customer_city,object,0,0.0,4119
4,customers,customer_state,object,0,0.0,27


In [9]:
column_profile.to_csv(
    OUTPUT_DIR / "column_profiling.csv",
    index=False
)

In [10]:
#Missing values analysis
missing_values = []

for table_name, df in tables.items():
    for column in df.columns:
        null_count = df[column].isna().sum()

        if null_count > 0:
            missing_values.append({
                "table": table_name,
                "column": column,
                "null_count": null_count,
                "null_pct": round(
                    null_count / len(df) * 100,
                    2
                )
            })

missing_values = pd.DataFrame(missing_values)

missing_values.sort_values(
    "null_pct",
    ascending=False
)

,table,column,null_count,null_pct
3,reviews,review_comment_title,87656,88.34
4,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
6,products,product_name_lenght,610,1.85
5,products,product_category_name,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
1,orders,order_delivered_carrier_date,1783,1.79
0,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


In [11]:
missing_values.to_csv(
    OUTPUT_DIR / "missing_values.csv",
    index=False
)

In [12]:
#duplicate analysis - complete-row duplicates:
duplicate_analysis = []

for table_name, df in tables.items():
    duplicate_analysis.append({
        "table": table_name,
        "duplicate_rows": df.duplicated().sum(),
        "duplicate_pct": round(
            df.duplicated().mean() * 100,
            2
        )
    })

duplicate_analysis = pd.DataFrame(duplicate_analysis)

duplicate_analysis

,table,duplicate_rows,duplicate_pct
0,customers,0,0.00
1,orders,0,0.00
2,order_items,0,0.00
3,payments,0,0.00
4,reviews,0,0.00
5,products,0,0.00
6,sellers,0,0.00
7,geolocation,261831,26.18
8,category_translation,0,0.00


In [ ]:
duplicate_analysis.to_csv(
    OUTPUT_DIR / "duplicate_analysis.csv",
    index=False
)

In [14]:
#Verify primary key

customers = tables["customers"]
customers["customer_id"].is_unique

orders = tables["orders"]
orders["order_id"].is_unique

products= tables["products"]
products["product_id"].is_unique

sellers = tables["sellers"]
sellers["seller_id"].is_unique

order_items= tables["order_items"]
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

payments= tables["payments"]
payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

reviews= tables["reviews"]
reviews.duplicated(
    subset=["order_id", "review_id"]
).sum()



np.int64(0)